# File này để fine tune model LLM

## File này bao gồm
- Tải model LLM
- Thiết lập tham số lượng tử hóa và tham số LoRA
- Tải dataset và định dạng lại dataset
- Khởi tạo hàm callback để thực thi Execution Accuracy sau N steps
- Lưu và chạy thử mô hình

In [1]:
import torch
import pandas as pd
from datasets import load_dataset
from transformers import (
    BitsAndBytesConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainerCallback,
    TrainerControl,
    TrainerState,
    TrainingArguments
)
from huggingface_hub import snapshot_download
from peft import PeftModel, LoraConfig
from trl import SFTTrainer, SFTConfig
import sqlite3
import sqlglot
import re
import os

root = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
SYSTEM_PROMPT = """You are a highly specialized NL2SQL engine. Your ONLY task is to translate natural language into accurate SQL queries based on the provided schema.

CRITICAL RULES:
1. Output EXACTLY ONE valid SQL query.
2. DO NOT include any greetings, explanations, or conversational text before or after the query.
3. DO NOT wrap the SQL query in markdown formatting blocks (e.g., strictly NO ```sql or ``` tags). Return raw text only.

Schema: \n"""

d:\python-workspace\NL2SQL-chat\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Tải model LLM

In [2]:
# snapshot_download(
#     repo_id="Qwen/Qwen2.5-Coder-3B-Instruct",
#     repo_type="model"
# )

## Thiết lập tham số lượng tử hóa và tham số LoRA

In [3]:
fp16 = False
bf16 = False
compute_dtype = None
if torch.cuda.is_available():
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=== Using bf16 data type ===")
        bf16 = True
        compute_dtype = torch.bfloat16
    else:
        fp16 = True
        compute_dtype = torch.float16
else:
    compute_dtype = torch.float32

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=False
)

peft_config = LoraConfig(
    task_type="CAUSAL_LM",
    lora_alpha=8,
    lora_dropout=0.1,
    r=4,
    bias="none"
)

=== Using bf16 data type ===


## Tải và định dạng dataset

In [4]:
dataset = load_dataset("b-mc2/sql-create-context")
print("Tải dataset thành công")
dataset

Tải dataset thành công


DatasetDict({
    train: Dataset({
        features: ['answer', 'question', 'context'],
        num_rows: 78577
    })
})

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "Qwen/Qwen2.5-Coder-3B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    device_map=device,
    dtype=compute_dtype
)
# model.config.use_cache = False
# model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-3B-Instruct")

Loading weights: 100%|██████████| 434/434 [00:20<00:00, 21.60it/s] 


In [6]:
def format_data(example, tokenizer):
    messages = [
        {
            "role": "system",
            "content": f"You are an AI assistant can code SQL perfectly. Write query accurately based on this schema:\n{example['context']} "
        },
        {
            "role": "user",
            "content": example['question']
        },
        {
            "role":"assistant",
            "content": example['answer']
        }
    ]
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False
    )
    return {"text": formatted_prompt}

dataset = dataset['train'].map(
    format_data,
    num_proc=8,
    fn_kwargs={"tokenizer":tokenizer}
)

In [7]:
dataset.remove_columns(['answer','question','context'])
for i in range(10):
    print(dataset['text'][i])

<|im_start|>system
You are an AI assistant can code SQL perfectly. Write query accurately based on this schema:
CREATE TABLE head (age INTEGER) <|im_end|>
<|im_start|>user
How many heads of the departments are older than 56 ?<|im_end|>
<|im_start|>assistant
SELECT COUNT(*) FROM head WHERE age > 56<|im_end|>

<|im_start|>system
You are an AI assistant can code SQL perfectly. Write query accurately based on this schema:
CREATE TABLE head (name VARCHAR, born_state VARCHAR, age VARCHAR) <|im_end|>
<|im_start|>user
List the name, born state and age of the heads of departments ordered by age.<|im_end|>
<|im_start|>assistant
SELECT name, born_state, age FROM head ORDER BY age<|im_end|>

<|im_start|>system
You are an AI assistant can code SQL perfectly. Write query accurately based on this schema:
CREATE TABLE department (creation VARCHAR, name VARCHAR, budget_in_billions VARCHAR) <|im_end|>
<|im_start|>user
List the creation year, name and budget of each department.<|im_end|>
<|im_start|>assi

In [8]:
dataset_split = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = dataset_split["train"]
eval_dataset  = dataset_split["test"]

print(f"Train: {len(train_dataset)} samples")
print(f"Eval : {len(eval_dataset)} samples")

Train: 70719 samples
Eval : 7858 samples


## Khởi tạo hàm thực thi Execution Accuracy

In [9]:
def _normalize_value(val):
    if val is None:
        return None
    if isinstance(val, float):
        return round(val, 4)
    if isinstance(val, str):
        stripped = val.strip()
        if re.match(r"^\d{4}-\d{2}-\d{2}$", stripped):
            return stripped
        return stripped.lower()
    return val

def _normalize_row(row: tuple):
    return tuple(_normalize_value(v) for v in row)

def _has_order_by(sql: str):
    return bool(re.search(r"\bORDER\s+BY\b", sql, re.IGNORECASE))

def execute_and_compare(
        pred_sql: str,
        ground_sql: str,
        cursor
):
    # Kiểm tra cú pháp
    try:
        sqlglot.parse_one(pred_sql, dialect="sqlite")
    except sqlglot.errors.SqlglotError:
        print("Lệnh SQL tạo ra bị lỗi cú pháp, trả về kết quả False")
        return False
    
    # Thực thi SQL ground truth
    try:
        cursor.execute(ground_sql)
        ground_rows = [_normalize_row(r) for r in cursor.fetchall()]
    except sqlite3.Error:
        print("Lệnh SQL thực tế bị lỗi")
        return False
    
    # Thực thi SQL pred
    try:
        cursor.execute(pred_sql)
        pred_rows = [_normalize_row(r) for r in cursor.fetchall()]
    except sqlite3.Error:
        print("Lệnh SQL bị ảo giác, trả về False")

    print(f"Dữ liệu từ SQL gold: {ground_rows}")
    print(f"Dữ liệu từ SQL pred: {pred_rows}")
    ground_has_order = _has_order_by(ground_sql)

    if ground_has_order:
        return pred_rows == ground_rows
    else:
        # Ít nhất một bên không quan tâm thứ tự → so sánh Set
        try:
            return set(map(frozenset, ground_rows)) == set(map(frozenset, pred_rows))
        except TypeError:
            # Fallback khi row chứa kiểu không hash được
            return sorted(ground_rows) == sorted(pred_rows)

In [10]:
# class ExecutionAccuracyCallback(TrainerCallback):
#     def __init__(
#         self,
#         eval_csv_path: str,
#         db_path: str,
#         system_prompt: str,
#         tokenizer,
#         query_column: str = "natural_language_query",
#         gold_sql_column: str = "gold_sql",
#         context_column: str = "context",
#         max_new_tokens: int = 200,
#     ) -> None:
#         self.eval_data      = pd.read_csv(eval_csv_path)
#         self.db_path        = db_path
#         self.system_prompt  = system_prompt
#         self.tokenizer      = tokenizer
#         self.query_col      = query_column
#         self.gold_sql_col   = gold_sql_column
#         self.context_col    = context_column
#         self.max_new_tokens = max_new_tokens

#     def _generate_sql(self, context: str, query: str, model):
#         messages = [
#             {"role": "system", "content": SYSTEM_PROMPT + context},
#             {"role": "user", "content": query}
#         ]
#         input_text =self.tokenizer.apply_chat_template(
#             messages,
#             tokenize=False,
#             add_generation_prompt=True
#         )
#         tokens = self.tokenizer(input_text, return_tensors="pt").to(model.device)
#         prompt_length = tokens["input_ids"].shape[1]

#         with torch.no_grad():
#             output = model.generate(
#                 **tokens,
#                 max_new_tokens=self.max_new_tokens,
#                 pad_token_id=self.tokenizer.eos_token_id,
#                 do_sample=False
#             )

#         generated_tokens = output[0][prompt_length:]
#         return self.tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

#     def on_evaluate(
#             self,
#             args: TrainingArguments,
#             state: TrainerState,
#             control: TrainerControl,
#             model,
#             **kwargs
#     ):
#         model.eval()
#         correct = 0
#         total = len(self.eval_data)

#         conn = sqlite3.connect(self.db_path)
#         cursor = conn.cursor()

#         try:
#             for _, row in self.eval_data.iterrows():
#                 query    : str = str(row[self.query_col])
#                 gold_sql : str = str(row[self.gold_sql_col]).strip()
#                 context  : str = str(row[self.context_col]).strip()
#                 print(f"Query: {query}")
#                 print(f"Gold SQL: {gold_sql}")
#                 print(f"Context: {context}")

#                 # Sinh pred_sql từ model
#                 # try:
#                 pred_sql = self._generate_sql(
#                     context=context,
#                     query=query,
#                     model=model
#                 )
#                 print(pred_sql)
#                 # except Exception:
#                 #     # Nếu generate thất bại, tính cặp này là sai
#                 #     continue

#                 # So sánh kết quả thực thi
#                 try:
#                     if execute_and_compare(pred_sql, gold_sql, cursor):
#                         correct += 1
#                 except Exception:
#                     # Phòng thủ: không để crash Trainer
#                     pass

#         finally:
#             conn.close()

#         execution_accuracy = correct / total if total > 0 else 0.0

#         # Log vào Trainer để hiển thị cùng loss và các metrics khác
#         metrics = {"execution_accuracy": round(execution_accuracy, 4)}
#         print(f"\n[EX Accuracy] Step {state.global_step} — "
#               f"{correct}/{total} = {execution_accuracy:.4f}\n")
#         # Ghi vào log_history của Trainer

#         state.log_history.append({
#             "step"                   : state.global_step,
#             "eval_execution_accuracy": round(execution_accuracy, 4),
#         })
        
# ea_eval_data_path = os.path.join(os.path.join(root, "data"), "ea_eval_data.csv")
# db_path = os.path.join(os.path.join(root, "data"), "EA_eval.db")
# ea_callback = ExecutionAccuracyCallback(
#     eval_csv_path=ea_eval_data_path,
#     db_path=db_path,
#     system_prompt=SYSTEM_PROMPT,
#     tokenizer=tokenizer
# )

## Fine tune model (CẤM CHẠY CELL DƯỚI)

In [11]:
train_args = SFTConfig(
    output_dir="../save_model_2",
    num_train_epochs=2, # Dừng theo epoch
    # max_steps=39500, # Dừng theo step
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    weight_decay=0.001,
    per_device_train_batch_size=4,
    logging_steps=500,
    fp16=fp16,
    bf16=bf16,
    dataset_text_field='text',
    packing=False,
    save_steps=500,
    save_total_limit=5,
    eval_strategy = "steps",
    eval_steps = 500,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    eval_dataset=eval_dataset,
    args=train_args,
    peft_config=peft_config
)
print("=== BẮT ĐẦU HUẤN LUYỆN ===")
trainer.train(resume_from_checkpoint=True)
print("=== HUẤN LUYỆN THÀNH CÔNG ===")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


=== BẮT ĐẦU HUẤN LUYỆN ===


	per_device_train_batch_size: 4 (from args) != 8 (from trainer_state.json)


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
stop

In [ ]:
adapter_path = r"..\save_model\checkpoint-39290"
finetuned_model = PeftModel.from_pretrained(model, adapter_path)
model.eval()
print("✅ Sẵn sàng! Mô hình đã được tải thành công.")

✅ Sẵn sàng! Mô hình đã được tải thành công.


In [ ]:
def generate_input(user_prompt, schema_db):
    messages = [
        {
            "role": "system",
            "content": f"You are an AI assistant can code SQL perfectly. Write query accurately based on this schema:\n{schema_db} "
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    return input_text

context_5 = """CREATE TABLE students (student_id INT PRIMARY KEY, name VARCHAR(100), major VARCHAR(50));
CREATE TABLE courses (course_id INT PRIMARY KEY, course_name VARCHAR(100), credits INT);
CREATE TABLE professors (prof_id INT PRIMARY KEY, name VARCHAR(100), department VARCHAR(50));
CREATE TABLE course_assignments (course_id INT, prof_id INT, semester VARCHAR(20));
CREATE TABLE enrollments (enrollment_id INT PRIMARY KEY, student_id INT, course_id INT, grade VARCHAR(2));"""
user_prompt_5 = "Get students name and major by fixing this query: SELECT * FROM students"

input_query = generate_input(user_prompt_5, context_5)
input_query

'<|im_start|>system\nYou are an AI assistant can code SQL perfectly. Write query accurately based on this schema:\nCREATE TABLE students (student_id INT PRIMARY KEY, name VARCHAR(100), major VARCHAR(50));\nCREATE TABLE courses (course_id INT PRIMARY KEY, course_name VARCHAR(100), credits INT);\nCREATE TABLE professors (prof_id INT PRIMARY KEY, name VARCHAR(100), department VARCHAR(50));\nCREATE TABLE course_assignments (course_id INT, prof_id INT, semester VARCHAR(20));\nCREATE TABLE enrollments (enrollment_id INT PRIMARY KEY, student_id INT, course_id INT, grade VARCHAR(2)); <|im_end|>\n<|im_start|>user\nGet students name and major by fixing this query: SELECT * FROM students<|im_end|>\n<|im_start|>assistant\n'

In [ ]:
with torch.no_grad():
    tokens = tokenizer(input_query, return_tensors="pt").to(device)
    outputs = model.generate(
        **tokens,
        max_new_tokens=256,
        temperature=0.1,
        pad_token_id = tokenizer.eos_token_id
    )
    outputs_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=False,
    )
    print(outputs)
    print(outputs_text.split("assistant")[-1][:-10])

tensor([[151644,   8948,    198,   2610,    525,    458,  15235,  17847,    646,
           2038,   7870,  13942,     13,   9645,   3239,  29257,   3118,    389,
            419,  10802,    510,  22599,  14363,   4143,    320,  12038,    842,
           9221,  37467,  12013,     11,    829,  37589,      7,     16,     15,
             15,    701,   3598,  37589,      7,     20,     15,   1106,  22599,
          14363,  13980,    320,  11856,    842,   9221,  37467,  12013,     11,
           3308,   1269,  37589,      7,     16,     15,     15,    701,  20141,
           9221,    317,  22599,  14363,  44624,    320,  21826,    842,   9221,
          37467,  12013,     11,    829,  37589,      7,     16,     15,     15,
            701,   9292,  37589,      7,     20,     15,   1106,  22599,  14363,
           3308,  20688,   1368,    320,  11856,    842,   9221,     11,   2778,
            842,   9221,     11,  33153,  37589,      7,     17,     15,   1106,
          22599,  14363,  51